# TTC Station Reliability

Which TTC stations are actually unreliable, and which just look bad because
they are busy? Delay counts are divided by scheduled trips to get a rate.

Run top to bottom. Tables persist in `ttc.duckdb`, so after a kernel restart
only the connection cell needs re-running.

## Setup — file inventory

24 files across `delay/`, `ridership/` and `schedules/`, mixed xlsx / csv / txt.

In [316]:
# Verifying data path

import os
import glob
import pandas as pd

paths = glob.glob("data/raw/**/*", recursive=True)

files = []
for item in paths:
    if os.path.isfile(item):
        files.append(item)

length = len(files)

print(files)
print(f"\n total files: {length}")

['data/raw\\delay\\TTC Subway Delay Data since 2025.csv', 'data/raw\\delay\\ttc-subway-delay-data-2018.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2019.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2020.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2021.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2022.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2023.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2024.xlsx', 'data/raw\\delay\\ttc-subway-delay-jan-2014-april-2017.xlsx', 'data/raw\\delay\\ttc-subway-delay-may-december-2017.xlsx', 'data/raw\\ridership\\1985-2019 Analysis of ridership.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2012-2013.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2014.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2015.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2016.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2017.xlsx', 'data/raw\\schedules\\agency.txt', 'data/raw\\schedules\\calendar.txt', 'data/raw\\schedules\\c

## Load GTFS

The schedule feed is the denominator, how many trains are scheduled to stop
at each station. 

This is chosen over ridership data as the datasets ends in 2019 which would not allow us to factor in COVID 19 disruptions and potential post COVID changes that might affect present day opertaions.

`route_type = 1` is subway, 
0 and 3 are persumably streetcarts and buses respectively

In [317]:
# Verifying routes 

import duckdb

con = duckdb.connect("ttc.duckdb")


con.sql("CREATE OR REPLACE TABLE routes AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/routes.txt')")


con.sql("SELECT COUNT(route_id) FROM routes GROUP BY route_type").df()



,count(route_id)
0,20
1,3
2,210


In [318]:
con.sql("SELECT route_id, route_short_name, route_long_name " \
"  FROM routes " \
"WHERE route_type = 1").df()

,route_id,route_short_name,route_long_name
0,1,1,Line 1 (Yonge-University)
1,2,2,Line 2 (Bloor - Danforth)
2,4,4,Line 4 (Sheppard)


### Load the remaining GTFS tables

`stop_times` is the bridge: it carries both `trip_id` and `stop_id`.

In [319]:
con.sql("CREATE OR REPLACE TABLE stops AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/stops.txt')")

con.sql("CREATE OR REPLACE TABLE stop_times AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/stop_times.txt')")

con.sql("CREATE OR REPLACE TABLE trips AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/trips.txt')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Denominator — scheduled trips per station

Filter `stop_times` to subway trips only, cutting 4.2M rows to ~150k.

In [320]:
# filter in only stop time data related to subway lines

con.sql("""
    CREATE OR REPLACE TABLE subway_stop_times AS
    SELECT stop_times.*, route_id FROM trips
    JOIN stop_times 
        ON trips.trip_id = stop_times.trip_id
    WHERE trips.route_id IN [1,2,4]
""")

con.sql("""
    SELECT * FROM subway_stop_times LIMIT 5
""").df()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,route_id
0,50659966,5:45:18,5:45:18,14945,1,None,0,0,NaN,1
1,50659966,5:47:44,5:47:44,15664,2,None,0,0,1.4442,1
2,50659966,5:50:22,5:50:22,15659,3,None,0,0,3.3901,1
3,50659966,5:52:34,5:52:34,15666,4,None,0,0,4.7396,1
4,50659966,5:54:26,5:54:26,15656,5,None,0,0,5.5524,1


### `parent_station` is null for TTC, so stations group on `stop_name`

In [321]:
con.sql("""
    SELECT * FROM stops LIMIT 5
""").df()

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
0,662,662,Danforth Rd at Kennedy Rd,None,43.714379,-79.260939,None,None,None,None,None,1
1,929,929,Davenport Rd at Bedford Rd,None,43.674448,-79.399659,None,None,None,None,None,1
2,940,940,Davenport Rd at Dupont St,None,43.675511,-79.401938,None,None,None,None,None,2
3,1871,1871,Davisville Ave at Cleveland St,None,43.702088,-79.378112,None,None,None,None,None,1
4,11700,11700,Disco Rd at Attwell Dr,None,43.701362,-79.594843,None,None,None,None,None,1


In [322]:
pd.set_option('display.max_rows', 20)

con.sql("""
    SELECT DISTINCT(route_type) FROM routes
""").df()

,route_type
0,0
1,1
2,3


### Merge Bloor-Yonge and normalise names

GTFS logs Bloor-Yonge as two stations — "Bloor" for the Line 2 platforms and
"Yonge" for Line 1. Merged here, giving 70 stations. Names are then uppercased
and stripped of " Station" to match the delay data's vocabulary.

Sanity check: the four interchanges show ~4,300 trips, ordinary stations
~2,100, Line 4 stops ~1,760.

In [ ]:
## fixing name duplicate i.e Bloor-Yonge

pd.set_option('display.max_rows', 80)

con.sql("""
    CREATE OR REPLACE TABLE station_trips AS
        SELECT 
            route_id AS line,
            SPLIT_PART(stop_name, ' -', 1) AS stations, 
            COUNT(*) AS scheduled_trips,
            AVG(stop_lat) AS lon,
            AVG(stop_lon) AS lat
        FROM stops
        JOIN subway_stop_times
            ON subway_stop_times.stop_id = stops.stop_id
        GROUP BY stations, line
        ORDER BY stations ASC
""")

con.sql("""
    CREATE OR REPLACE TABLE station_geo_info AS
        WITH 
        s1 AS (
            SELECT * REPLACE (SPLIT_PART(stations, ' Station' , 1) AS stations) FROM station_trips),
        s2 AS (
            SELECT * REPLACE (UPPER(stations) AS stations) FROM s1),
        s3 AS (
            SELECT * RENAME (stations AS station) FROM s2),
        s4 AS (
            SELECT * REPLACE (
                CASE WHEN station IN ('BLOOR', 'YONGE') THEN 'BLOOR-YONGE'
                ELSE station END AS station)
            FROM s3)
    SELECT * FROM s4
""")

con.sql("""
    CREATE OR REPLACE TABLE station_trips_clean AS
        SELECT * EXCLUDE(lat, lon) FROM station_geo_info
""")


con.sql("""SELECT * FROM station_geo_info""").df()


,line,station,scheduled_trips,lon,lat
0,2,BATHURST,2137,43.665798,-79.411443
1,2,BAY,2136,43.669998,-79.390942
2,4,BAYVIEW,1755,43.766912,-79.386717
3,4,BESSARION,1758,43.769249,-79.376329
4,1,BLOOR-YONGE,2196,43.670546,-79.385654
5,2,BROADVIEW,2133,43.676698,-79.358840
6,2,CASTLE FRANK,2133,43.673798,-79.368940
7,1,CEDARVALE,2181,43.700002,-79.436492
8,2,CHESTER,2133,43.678296,-79.352520
9,2,CHRISTIE,2139,43.664298,-79.418144


In [324]:
pd.set_option('display.max_rows', 20)

con.sql("""
CREATE OR REPLACE TABLE station_lines AS
    SELECT station,
           COUNT(*) AS n_lines,
           ANY_VALUE(line) AS sole_line
    FROM station_trips_clean
    GROUP BY station
""")

con.sql("""
    SELECT * FROM station_lines
""").df()

,station,n_lines,sole_line
0,COLLEGE,1,1
1,DAVISVILLE,1,1
2,EGLINTON,1,1
3,CEDARVALE,1,1
4,JANE,1,2
5,KIPLING,1,2
6,OSGOODE,1,1
7,RUNNYMEDE,1,2
8,SPADINA,2,2
9,ST GEORGE,2,2


## Load delay data

10 files, Jan 2014 – Jun 2026. Column schema is stable across all of them;
the csv adds an `_id` column from CKAN.

In [325]:
## Verifying delay datas

paths = glob.glob("data/raw/delay/**")

for f in paths:
    if os.path.splitext(f)[1].lower() == '.xlsx':
        df = pd.read_excel(f, nrows=0)
    else:
        df = pd.read_csv(f, nrows=0)
    print(f, df.columns.tolist())

path = "data/raw/delay/ttc-subway-delay-jan-2014-april-2017.xlsx"
sheets = pd.read_excel(path, sheet_name=None)
print(sheets.keys())

print(sheets['Incidents']['Date'].min())
print(sheets['Incidents']['Date'].max())

data/raw/delay\TTC Subway Delay Data since 2025.csv ['_id', 'Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2018.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2019.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2020.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2021.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2022.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2023.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehi

In [326]:
paths = glob.glob("data/raw/delay/**")

dfs = []

for f in paths:
    if os.path.splitext(f)[1].lower() == '.xlsx':
        dfs.extend(pd.read_excel(f, sheet_name=None).values())
    else:
        dfs.append(pd.read_csv(f))

all_delays = pd.concat(dfs, ignore_index=True)

## all_delays.shape

(all_delays.dtypes.to_string())

## print(all_delays['Date'].max())
## print(all_delays['Date'].min())

'_id          float64\nDate          object\nTime             str\nDay              str\nStation          str\nCode             str\nMin Delay      int64\nMin Gap        int64\nBound            str\nLine             str\nVehicle        int64'

### Station name concentration

1,853 distinct station values against a true count of 70. The top 70 cover
~90% of rows and the top 150 ~98%, so the ~1,700-value tail is almost all
singletons — which is what makes exclusion defensible rather than mapping.

In [327]:
pd.set_option('display.max_rows', 100)

con.sql("""
CREATE OR REPLACE TABLE raw_delays AS
SELECT * FROM all_delays 
""")

n = con.sql("SELECT COUNT(DISTINCT Station) AS n FROM raw_delays").fetchone()[0]

print(f"distinct stations: {n}\n")

con.sql("""
WITH ranked AS (
    SELECT
        Station, 
        COUNT(Station) AS count,
        count / SUM(COUNT(*)) OVER () AS ratio,
        SUM(COUNT(*)) OVER (ORDER BY COUNT(*) DESC) as r_total,
        r_total / SUM(COUNT(*)) OVER () AS r_ratio,
        ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS r_num
    FROM raw_delays
    GROUP BY Station
    ORDER BY Count DESC
)
SELECT * FROM ranked WHERE r_num IN (70,150)
""").df()



distinct stations: 2178



,Station,count,ratio,r_total,r_ratio,r_num
0,RUNNYMEDE STATION,1451,0.005517,236483.0,0.899096,70
1,KIPLING STATION (APPRO,24,0.000091,258660.0,0.983412,150


### Raw delay table

All 186,916 rows as concatenated, before any cleaning.

In [328]:
con.sql("""
SELECT * FROM all_delays
""").df()

,_id,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
0,1.0,2025-01-01,02:10,Wednesday,BATHURST STATION,MUSAN,5,9,E,BD,5227
1,2.0,2025-01-01,02:30,Wednesday,DUNDAS STATION,MUIRS,0,0,NaN,YU,0
2,3.0,2025-01-01,02:32,Wednesday,BROADVIEW STATION,PUMST,0,0,E,BD,0
3,4.0,2025-01-01,02:58,Wednesday,KEELE STATION,EUSC,0,0,W,BD,5293
4,5.0,2025-01-01,02:58,Wednesday,COXWELL STATION,SUAE,0,0,NaN,BD,0
5,6.0,2025-01-01,07:59,Wednesday,DONLANDS STATION,TUNOA,5,10,W,BD,5000
6,7.0,2025-01-01,08:13,Wednesday,BLOOR STATION,PUTR,9,0,S,YU,5836
7,8.0,2025-01-01,08:15,Wednesday,BLOOR STATION,PUTR,5,0,N,YU,5706
8,9.0,2025-01-01,09:14,Wednesday,FINCH STATION,SUDP,0,0,S,YU,5596
9,10.0,2025-01-01,09:22,Wednesday,BATHURST STATION,SUUT,0,0,E,BD,5312


## Clean layer

Each CTE applies one rule. Order matters:

- **s1** drops Line 3 (SRT) — closed 2023, absent from GTFS, no denominator.
  Must come before the suffix strip or Kennedy SRT merges into Kennedy.
- **s3** splits at `' STATION'`, which also removes trailing descriptors
  like `'ROYAL YORK STATION (AP'`.
- **s10** anchors `DUNDAS` → `TMU`, since `DUNDAS WEST` is a different station.
- **s19/s20** reconcile Date, which parses as a timestamp from xlsx and as
  text from the csv, then combine it with Time.
- **s21** is the exclusion step: the join drops yards, hostlers, carhouses,
  wyes, portals, two-station segments and line-level records — ~15k rows (10%),
  none of which can be attributed to a single station.

In [329]:
pd.set_option('display.max_rows', 25)

con.sql("""
CREATE OR REPLACE TABLE clean_delays_all AS 
    WITH 
    s1 AS (
        SELECT * FROM raw_delays WHERE Line IS DISTINCT FROM 'SRT'),
    s1a AS (
        SELECT * FROM s1 WHERE NOT station LIKE '% TO %' AND station NOT LIKE '%SHUTTLE'),
    s2 AS (
        SELECT * REPLACE (SPLIT_PART(Station, ' STATION', 1) AS Station) FROM s1a),
    s3 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, ' (BD|YU|YUS|SRT)$', '') AS Station) FROM s2),
    s4 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, 'SHEPPARDSTATION$', 'SHEPPARD-YONGE') AS Station) FROM s3),
    s5 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, 'BLOOR YONGE', 'BLOOR-YONGE') AS Station) FROM s4),
    s6 AS (
        SELECT * RENAME (Station AS station) FROM s5),
    s7 AS (
        SELECT * REPLACE (REPLACE(station, '.', '') AS station) FROM s6),
    s8 AS (
        SELECT * REPLACE (REPLACE(station, 'EGLINTON WEST', 'CEDARVALE') AS station) FROM s7),
    s9 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^DUNDAS$', 'TMU') AS station) FROM s8),
    s11 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^BLOOR$', 'BLOOR-YONGE') AS station) FROM s9),
    s12 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^YONGE$', 'BLOOR-YONGE') AS station) FROM s11),
    s13 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^SHEPPARD$', 'SHEPPARD-YONGE') AS station) FROM s12),
    s14 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^VAUGHAN MC$', 'VAUGHAN METROPOLITAN CENTRE') AS station) FROM s13),
    s15 AS (
        SELECT * REPLACE (REPLACE(station, 'CTR', 'CENTRE') AS station) FROM s14),
    s16 AS (
        SELECT * REPLACE (REPLACE(station, 'VMC', 'VAUGHAN METROPOLITAN CENTRE') AS station) FROM s15),
    s17 AS (
        SELECT * REPLACE (REPLACE(station, 'YONGE SHP', 'SHEPPARD-YONGE') AS station) FROM s16),
    s18 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, ' STATIO$', '') AS station) FROM s17),
    s19 AS (
        SELECT * EXCLUDE(Date), SPLIT_PART(Date, ' ', 1) AS Date_normalized FROM s18),
    s20 AS (
        SELECT * FROM s19 WHERE Line NOT LIKE '%SHUTTLE%'),
    s21 AS (
        SELECT *, 
                CASE Line 
                    WHEN 'YUS' THEN 1
                    WHEN 'YU' THEN 1
                    WHEN 'BD' THEN 2
                    WHEN 'SHP' THEN 4
                    WHEN 'SHEP' THEN 4
                END AS route
            FROM s20),
    s22 AS (
        SELECT * EXCLUDE(_id, Time, Date_normalized), STRPTIME(Date_normalized || ' ' || Time, '%Y-%m-%d %H:%M') AS Timestamp FROM s21)
SELECT * FROM s22
""")

con.sql("""
    CREATE OR REPLACE TABLE clean_delays_1 AS 
        SELECT * EXCLUDE(t2.station, scheduled_trips, Line) FROM clean_delays_all t1
        LEFT JOIN station_trips_clean t2
            ON t1.station = t2.station AND t1.route = t2.line
        WHERE t2.station IS NOT NULL
""")

result = con.sql("""
    SELECT * FROM clean_delays_1
""").df()

result



,Day,station,Code,Min Delay,Min Gap,Bound,Vehicle,route,Timestamp
0,Wednesday,ST PATRICK,TUSC,0,0,S,5986,1,2017-08-09 00:47:00
1,Wednesday,MAIN STREET,SUAP,0,0,E,0,2,2017-08-09 01:20:00
2,Wednesday,DUNDAS WEST,SUUT,10,0,W,5313,2,2017-08-09 01:58:00
3,Wednesday,LAWRENCE WEST,MUNCA,0,0,NaN,0,1,2017-08-09 03:32:00
4,Wednesday,SPADINA,PUTO,0,0,N,6071,1,2017-08-09 05:55:00
5,Wednesday,EGLINTON,EUDO,4,8,N,5111,1,2017-08-09 06:23:00
6,Wednesday,KENNEDY,MUSC,0,0,NaN,5212,2,2017-08-09 06:53:00
7,Wednesday,BLOOR-YONGE,EUSC,0,0,W,5191,2,2017-08-09 07:03:00
8,Wednesday,ST GEORGE,EUSC,0,0,W,5191,2,2017-08-09 07:08:00
9,Wednesday,SHEPPARD-YONGE,SUDP,3,5,S,6026,1,2017-08-09 07:51:00


### Inconsistent station line 

some stations was assigned wrong or null route value

In [330]:
pd.set_option('display.max_rows', 25)

con.sql("""
    WITH table1 AS (
        SELECT * EXCLUDE (t2.station) FROM clean_delays_all t1
        JOIN ( SELECT DISTINCT station FROM station_trips_clean) t2
            ON t1.station = t2.station
        WHERE t2.station IS NOT NULL
    )

    SELECT t1.station, route, COUNT(t1.station) AS count FROM table1 t1
    LEFT JOIN station_trips_clean t2
        ON t1.route = t2.Line AND t1.station = t2.station
    WHERE t2.Line IS NULL
    GROUP BY t1.station, route
    ORDER BY count DESC
""").df()

,station,route,count
0,WARDEN,1,120
1,WARDEN,<NA>,35
2,KENNEDY,1,28
3,KIPLING,1,22
4,SHERBOURNE,1,13
5,TMU,2,12
6,BAY,1,11
7,ROYAL YORK,1,11
8,FINCH,2,11
9,DONLANDS,1,10


### Repair table

repair broken or wrong route number for stations

In [331]:
con.sql("""
    CREATE OR REPLACE TABLE route_repair AS (
        WITH table1 AS (
            SELECT * EXCLUDE (t2.station) FROM clean_delays_all t1
            JOIN ( SELECT DISTINCT station FROM station_trips_clean) t2
                ON t1.station = t2.station
            WHERE t2.station IS NOT NULL
        )

        SELECT t1.* EXCLUDE(Line) REPLACE (t3.sole_line AS route)
        FROM table1 t1
        LEFT JOIN station_trips_clean t2
            ON t1.route = t2.Line AND t1.station = t2.station
        LEFT JOIN station_lines t3
            ON t1.station = t3.station
        WHERE t2.Line IS NULL AND n_lines = 1
    )
""")

con.sql("""
    SELECT * FROM route_repair
""").df()

,Day,station,Code,Min Delay,Min Gap,Bound,Vehicle,route,Timestamp
0,Monday,CASTLE FRANK,MUIS,0,0,NaN,7776,2,2017-10-09 17:18:00
1,Thursday,GLENCAIRN,MUSC,0,0,W,5288,1,2017-10-12 14:34:00
2,Wednesday,VICTORIA PARK,MUSC,0,0,W,5240,2,2017-11-01 00:52:00
3,Wednesday,DONLANDS,PUSSW,0,0,W,5254,2,2017-11-01 06:23:00
4,Wednesday,PAPE,MUSC,0,0,W,5190,2,2017-11-01 07:25:00
5,Wednesday,BROADVIEW,SUAE,0,0,NaN,0,2,2017-11-01 08:32:00
6,Wednesday,VICTORIA PARK,SUDP,0,0,W,5068,2,2017-11-01 10:33:00
7,Wednesday,OLD MILL,MUO,0,0,W,5322,2,2017-11-01 11:55:00
8,Wednesday,HIGH PARK,MUIS,0,0,NaN,0,2,2017-11-01 15:03:00
9,Wednesday,VICTORIA PARK,PUTIJ,7,9,W,5312,2,2017-11-01 16:18:00


In [332]:
con.sql("""
    CREATE OR REPLACE TABLE clean_delays AS (
        SELECT * FROM clean_delays_1
        UNION ALL 
        SELECT * FROM route_repair
    )
""")

con.sql("""
    SELECT * FROM clean_delays
""").df()

,Day,station,Code,Min Delay,Min Gap,Bound,Vehicle,route,Timestamp
0,Wednesday,ST PATRICK,TUSC,0,0,S,5986,1,2017-08-09 00:47:00
1,Wednesday,MAIN STREET,SUAP,0,0,E,0,2,2017-08-09 01:20:00
2,Wednesday,DUNDAS WEST,SUUT,10,0,W,5313,2,2017-08-09 01:58:00
3,Wednesday,LAWRENCE WEST,MUNCA,0,0,NaN,0,1,2017-08-09 03:32:00
4,Wednesday,SPADINA,PUTO,0,0,N,6071,1,2017-08-09 05:55:00
5,Wednesday,EGLINTON,EUDO,4,8,N,5111,1,2017-08-09 06:23:00
6,Wednesday,KENNEDY,MUSC,0,0,NaN,5212,2,2017-08-09 06:53:00
7,Wednesday,BLOOR-YONGE,EUSC,0,0,W,5191,2,2017-08-09 07:03:00
8,Wednesday,ST GEORGE,EUSC,0,0,W,5191,2,2017-08-09 07:08:00
9,Wednesday,SHEPPARD-YONGE,SUDP,3,5,S,6026,1,2017-08-09 07:51:00


### After cleaning

In [333]:
pd.set_option('display.max_rows', 100)

con.sql("""
WITH ranked AS (
    SELECT
        Station, 
        COUNT(Station) AS count,
        count / SUM(COUNT(*)) OVER () AS ratio,
        SUM(COUNT(*)) OVER (ORDER BY COUNT(*) DESC) as r_total,
        r_total / SUM(COUNT(*)) OVER () AS r_ratio,
        ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS r_num
    FROM clean_delays_all
    GROUP BY Station
    ORDER BY Count DESC
)
SELECT * FROM ranked WHERE r_num IN (70,150)
""").df()

,station,count,ratio,r_total,r_ratio,r_num
0,HIGHWAY 407,1018,0.004019,244806.0,0.966406,70
1,LINE 2- BLOOR DANFORTH,8,0.000032,252274.0,0.995887,150


### Every row parsed — no null timestamps

In [334]:
con.sql("""
SELECT COUNT(*) FROM clean_delays
WHERE Timestamp IS NULL
""").df()

,count_star()
0,0


### `clean_delays` after all rules

In [335]:
pd.set_option('display.min_rows', 5)

con.sql("""
SELECT *
FROM clean_delays
""").df()

,Day,station,Code,Min Delay,Min Gap,Bound,Vehicle,route,Timestamp
0,Wednesday,ST PATRICK,TUSC,0,0,S,5986,1,2017-08-09 00:47:00
1,Wednesday,MAIN STREET,SUAP,0,0,E,0,2,2017-08-09 01:20:00
...,...,...,...,...,...,...,...,...,...
238085,Monday,ST PATRICK,SUO,5,8,S,5511,1,2022-10-17 20:35:00
238086,Wednesday,ROSEDALE,SUDP,10,13,N,5461,1,2022-10-19 08:29:00


## Exclusion checks

Segments name two stations, so they have no single denominator. Under 1% of
rows, too small to bias the ranking.

In [336]:
pd.set_option('display.min_rows', 10)
pd.set_option('display.max_rows', 10)

con.sql("""
SELECT t1.station, route, COUNT(*) AS count
FROM clean_delays_all AS t1 
LEFT JOIN station_trips_clean t2 
    ON t1.station = t2.station
WHERE t2.station IS NULL
GROUP BY t1.station, route 
ORDER BY count DESC
""").df()




,station,route,count
0,YONGE UNIVERSITY LINE,1,3210
1,BLOOR DANFORTH SUBWAY,2,2564
2,YONGE-UNIVERSITY AND B,<NA>,1907
3,YUS/BD/SHEPPARD SUBWAY,<NA>,777
4,GREENWOOD YARD,2,703
...,...,...,...
904,BEDFORD SUBSTATION,2,1
905,SHEPPARD TAIL TRACK #2,4,1
906,SHEPPARD-YONGE AND ST,1,1
907,1900 YONGE STREET,1,1


### Delay rows per matched station

Coverage check: every one of the 70 GTFS stations has delay rows.

In [337]:
con.sql("""
SELECT t2.station, COUNT(*) AS count
FROM clean_delays AS t1 
RIGHT JOIN station_trips_clean t2 
    ON t1.station = t2.station
WHERE t1.station IS NOT NULL
GROUP BY t2.station
ORDER BY station ASC
""").df()


,station,count
0,BATHURST,2716
1,BAY,1957
2,BAYVIEW,1178
3,BESSARION,695
4,BLOOR-YONGE,25456
...,...,...
65,WILSON,5651
66,WOODBINE,2570
67,YORK MILLS,3590
68,YORK UNIVERSITY,652


### Rows with no matching station

In [338]:
con.sql("""
SELECT COUNT(*) AS count
FROM clean_delays_all AS t1 
LEFT JOIN station_trips_clean t2 
    ON t1.station = t2.station
WHERE t2.station IS NULL
""").df()

,count
0,15202


### Zero-delay rows

65% of rows have `Min Delay = 0` — logged incidents with no measurable delay.
Nearly all also have zero gap, meaning no service impact, so they are excluded
from the metrics below.

In [339]:
pd.set_option('display.min_rows', 30)
pd.set_option('display.max_rows', 30)

con.sql("""
SELECT COUNT(*) FROM clean_delays 
WHERE "Min Delay" > 0
""").df()

,count_star()
0,86291


In [340]:
con.sql("""
SELECT COUNT(*) FROM clean_delays 
WHERE "Min Gap" > 0
""").df()

,count_star()
0,83619


## Analysis — full range

`delay_rate` = incidents / scheduled trips. Also computes total and average
delay minutes, plus p50 and p95 for severity.

In [341]:
## raw unreliability number

pd.set_option('display.min_rows', 40)
pd.set_option('display.max_rows', 40)

con.sql("""
CREATE OR REPLACE TABLE station_unreliability AS 
    WITH 
    s1 AS (
        SELECT station, 
            COUNT("Min Delay") AS delays, 
            SUM("Min Delay") AS total_delay, 
            AVG("Min Delay") AS avg_delay,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "Min Delay") AS p50,
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "Min Delay") AS p95,
            route
        FROM clean_delays 
        WHERE "Min Delay" > 0
        GROUP BY station, route),
    s2 AS (
        SELECT 
            t1.station, 
            delays / scheduled_trips AS delay_rate, 
            delays, total_delay, avg_delay,
            p50,
            p95,
            route
        FROM s1 t1 
        JOIN station_trips_clean t2 
            ON t1.station = t2.station)
    SELECT *, 
        RANK() OVER (ORDER BY delay_rate DESC) AS adjusted_rank, 
        RANK() OVER (ORDER BY delays DESC) AS raw_rank,
        raw_rank - adjusted_rank AS delta
    FROM s2
""")

### Extreme delays

A handful of multi-hour records. The Jan 2026 and Feb 2025 clusters span
multiple lines on consecutive days, so these are real system-wide disruptions
rather than entry errors. Retained — median and p95 are used instead of the
mean so they do not distort the ranking.

In [342]:
con.sql("""
SELECT station, route, "Min Delay", Timestamp FROM clean_delays
WHERE "Min Delay" > 400
""").df()

,station,route,Min Delay,Timestamp
0,EGLINTON,1,807,2025-02-16 09:18:00
1,SHEPPARD WEST,1,900,2025-02-16 11:03:00
2,VICTORIA PARK,2,661,2026-01-25 15:21:00
3,GLENCAIRN,1,622,2026-01-25 16:12:00
4,KIPLING,2,441,2026-01-25 18:32:00
5,WOODBINE,2,827,2026-01-26 05:50:00
6,ST CLAIR WEST,1,505,2026-01-26 06:00:00
7,ISLINGTON,2,481,2026-01-26 06:39:00
8,OLD MILL,2,484,2026-04-07 05:46:00
9,KENNEDY,2,515,2018-10-20 17:44:00


### Data range Verification

In [343]:
con.sql("""
SELECT MAX(Timestamp), MIN(Timestamp) FROM clean_delays
""").df()

,"max(""Timestamp"")","min(""Timestamp"")"
0,2026-06-30 23:55:00,2014-01-01 00:21:00


## Analysis — 2018 onward

The Vaughan extension opened December 2017, so its six stations have delays
for only part of the full window while carrying the same denominator. Their
rates are understated. This version is the primary result.

In [344]:
## with 2017 line 1 extension sensitivity consideration

pd.set_option('display.min_rows', 40)
pd.set_option('display.max_rows', 40)

con.sql("""
CREATE OR REPLACE TABLE station_unreliability_2018 AS 
    WITH 
    s1 AS (
        SELECT station, 
            COUNT("Min Delay") AS delays, 
            SUM("Min Delay") AS total_delay, 
            AVG("Min Delay") AS avg_delay,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "Min Delay") AS p50,
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "Min Delay") AS p95,
            route
        FROM clean_delays 
        WHERE "Min Delay" > 0 AND Timestamp >= '2018-01-01'
        GROUP BY station, route),
    s2 AS (
        SELECT 
            t1.station, 
            delays / scheduled_trips AS delay_rate, 
            delays, total_delay, avg_delay,
            p50,
            p95,
            route
        FROM s1 t1 
        JOIN station_trips_clean t2 
            ON t1.station = t2.station)
    SELECT *, 
        RANK() OVER (ORDER BY delay_rate DESC) AS adjusted_rank, 
        RANK() OVER (ORDER BY delays DESC) AS raw_rank,
        raw_rank - adjusted_rank AS delta
    FROM s2
""")

## GTFS service calendar

Loaded but not currently used — the denominator counts trips defined in
the feed without weighting them by how many days each service actually runs.

In [345]:
con.sql("""
CREATE OR REPLACE TABLE calendar AS
    SELECT * FROM read_csv_auto('data/raw/schedules/calendar.txt')
""")

con.sql("""
SELECT * FROM calendar
""").df()

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,1,1,1,1,1,1,0,0,20260726,20260905
1,2,0,0,0,0,0,1,0,20260726,20260905
2,3,0,0,0,0,0,0,1,20260726,20260905
3,4,0,0,0,0,0,0,0,20260726,20260905
4,501,0,0,0,0,0,0,0,20260726,20260905
5,6701,0,0,0,0,0,0,0,20260726,20260905
6,4401,0,0,0,0,0,0,0,20260726,20260905
7,4501,0,0,0,0,0,0,0,20260726,20260905
8,7001,0,0,0,0,0,0,0,20260726,20260905
9,6702,0,0,0,0,0,0,0,20260726,20260905


### Sensitivity: full range vs 2018

Positive `adjusted_diff` means the station ranked better in the full-range
version than it should have. Highway 407 moves 21 places, Finch West 16 —
confirming the bias.

In [346]:
con.sql("""
SELECT 
    t1.station, 
    t1.raw_rank - t2.raw_rank AS raw_diff,
    t1.adjusted_rank - t2.adjusted_rank AS adjusted_diff,
    t1.route
FROM station_unreliability t1 
JOIN station_unreliability_2018 t2
    ON t1.station = t2.station
ORDER BY adjusted_diff DESC
""").df()

## the more positive the worst the delay

,station,raw_diff,adjusted_diff,route
0,HIGHWAY 407,16,18,1
1,SHEPPARD-YONGE,1,17,1
2,SHEPPARD-YONGE,-2,14,4
3,FINCH WEST,10,11,1
4,ST GEORGE,8,10,2
5,BLOOR-YONGE,8,9,2
6,ST GEORGE,8,9,2
7,BLOOR-YONGE,8,8,2
8,BLOOR-YONGE,8,8,2
9,ROSEDALE,6,7,1


## Top 10 — adjusted rank, full range

In [347]:
con.sql("""
SELECT station, route, adjusted_rank FROM station_unreliability
ORDER BY adjusted_rank ASC
LIMIT 10
""").df()


,station,route,adjusted_rank
0,KENNEDY,2,1
1,FINCH,1,2
2,KIPLING,2,3
3,EGLINTON,1,4
4,VAUGHAN METROPOLITAN CENTRE,1,5
5,WILSON,1,6
6,BLOOR-YONGE,1,7
7,BLOOR-YONGE,1,8
8,SHEPPARD WEST,1,9
9,DAVISVILLE,1,10


## Top 10 — adjusted rank, 2018 onward

In [348]:
con.sql("""
SELECT station, route, adjusted_rank FROM station_unreliability_2018
ORDER BY adjusted_rank ASC
LIMIT 10
""").df()

,station,route,adjusted_rank
0,VAUGHAN METROPOLITAN CENTRE,1,1
1,FINCH,1,2
2,KENNEDY,2,3
3,KIPLING,2,4
4,EGLINTON,1,5
5,WILSON,1,6
6,BLOOR-YONGE,1,7
7,BLOOR-YONGE,1,8
8,SHEPPARD WEST,1,9
9,COXWELL,2,10


## Top 10 — raw count, full range

In [349]:
con.sql("""
SELECT station, route, raw_rank FROM station_unreliability
ORDER BY raw_rank ASC
LIMIT 10
""").df()

,station,route,raw_rank
0,KENNEDY,2,1
1,FINCH,1,2
2,KIPLING,2,3
3,EGLINTON,1,4
4,VAUGHAN METROPOLITAN CENTRE,1,5
5,WILSON,1,6
6,BLOOR-YONGE,1,7
7,BLOOR-YONGE,1,7
8,SHEPPARD WEST,1,9
9,DAVISVILLE,1,10


## Top 10 — raw count, 2018 onward

Comparing this against the adjusted list gives the headline: the stations that
drop out are all interchanges, which carry roughly double the scheduled trips.

In [350]:
con.sql("""
SELECT station, route, raw_rank FROM station_unreliability_2018
ORDER BY raw_rank ASC
LIMIT 10
""").df()

,station,route,raw_rank
0,VAUGHAN METROPOLITAN CENTRE,1,1
1,FINCH,1,2
2,KENNEDY,2,3
3,KIPLING,2,4
4,EGLINTON,1,5
5,WILSON,1,6
6,BLOOR-YONGE,1,7
7,BLOOR-YONGE,1,7
8,SHEPPARD WEST,1,9
9,DAVISVILLE,1,10


### Denominator recap

In [351]:
con.sql("""
    SELECT * FROM station_trips_clean
""").df()

,line,station,scheduled_trips
0,2,BATHURST,2137
1,2,BAY,2136
2,4,BAYVIEW,1755
3,4,BESSARION,1758
4,1,BLOOR-YONGE,2196
5,2,BROADVIEW,2133
6,2,CASTLE FRANK,2133
7,1,CEDARVALE,2181
8,2,CHESTER,2133
9,2,CHRISTIE,2139


## Confidence intervals

A parametric bootstrap. Delay counts should follow a Poisson distribution, and
a Poisson's variance equals its mean, so the observed count is the only
parameter needed. Each station gets 5,000 simulated counts, converted to rates,
with the 2.5th and 97.5th percentiles taken as a 95% interval.

`worse_than_baseline` checks whether even the low end of a station's interval
still sits above the network average rate.

One limitation. Poisson assumes events happen independently, but delays
clusters, since a single incident often generates several records. That means
the real counts are overdispersed and these intervals come out narrower than
they should be. A negative binomial model would fix this.

In [352]:
df = con.sql("""
    SELECT t1.station, route, delays, scheduled_trips
    FROM station_unreliability_2018 t1 
    JOIN station_trips_clean t2 ON t1.station = t2.station
""").df()

import numpy as np
rng = np.random.default_rng(1)

sims = rng.poisson(df['delays'].values,size=(5000, len(df)))

rates = sims / df['scheduled_trips'].values

df['ci_low'], df['ci_high'] = np.percentile(rates, [2.5, 97.5], axis=0)
df['rate'] = df['delays'] / df['scheduled_trips']

baseline = df['delays'].sum() / df['scheduled_trips'].sum()
df['worse_than_baseline'] = df['ci_low'] > baseline

df.sort_values('rate', ascending=False).head(15)

,station,route,delays,scheduled_trips,ci_low,ci_high,rate,worse_than_baseline
0,VAUGHAN METROPOLITAN CENTRE,1,3032,2154,1.357939,1.458229,1.407614,True
1,FINCH,1,2924,2224,1.267086,1.362860,1.314748,True
2,KENNEDY,2,2754,2158,1.229831,1.322984,1.276182,True
3,KIPLING,2,2488,2155,1.108585,1.200000,1.154524,True
4,EGLINTON,1,2462,2223,1.062978,1.149798,1.107512,True
5,WILSON,1,2225,2181,0.978450,1.062815,1.020174,True
6,BLOOR-YONGE,1,1928,2136,0.862828,0.942896,0.902622,True
7,BLOOR-YONGE,1,1928,2136,0.862828,0.943352,0.902622,True
83,BLOOR-YONGE,1,1928,2196,0.838798,0.917122,0.877960,True
82,BLOOR-YONGE,1,1928,2196,0.839709,0.917577,0.877960,True


## Export

`station_data_final` joins the 2018 metrics to the bootstrap intervals.
`rank_comparison` is the same ranks in long format, which is the shape
Tableau needs for a slope chart.

In [353]:
con.sql("""
    CREATE OR REPLACE TABLE station_ci AS
        SELECT * FROM df
""")

con.sql("""
    CREATE OR REPLACE TABLE station_data_final AS
        SELECT * EXCLUDE(t2.station, t2.delays, rate) FROM station_unreliability_2018 t1 
        JOIN station_ci t2 
            ON t1.station = t2.station 
""")

con.sql("""
    SELECT * FROM station_data_final
""").df()


con.sql("""
    COPY station_data_final TO 'output/station_final.csv' (HEADER, DELIMITER ',')
""")

In [354]:
con.sql("""
    SELECT * FROM station_data_final
""").df()

,station,delay_rate,delays,total_delay,avg_delay,p50,p95,route,adjusted_rank,raw_rank,delta,route_1,scheduled_trips,ci_low,ci_high,worse_than_baseline
0,VAUGHAN METROPOLITAN CENTRE,1.407614,3032,14140.0,4.663588,3.0,9.0,1,1,1,0,1,2154,1.357939,1.458229,True
1,FINCH,1.314748,2924,15404.0,5.268126,4.0,11.0,1,2,2,0,1,2224,1.267086,1.362860,True
2,KENNEDY,1.276182,2754,15916.0,5.779230,4.0,14.0,2,3,3,0,2,2158,1.229831,1.322984,True
3,KIPLING,1.154524,2488,14643.0,5.885450,4.0,13.0,2,4,4,0,2,2155,1.108585,1.200000,True
4,EGLINTON,1.107512,2462,16672.0,6.771730,5.0,15.0,1,5,5,0,1,2223,1.062978,1.149798,True
5,WILSON,1.020174,2225,13220.0,5.941573,4.0,14.0,1,6,6,0,1,2181,0.978450,1.062815,True
6,BLOOR-YONGE,0.902622,1928,12877.0,6.678942,4.0,17.0,1,7,7,0,2,2196,0.537341,0.599727,True
7,BLOOR-YONGE,0.877960,1928,12877.0,6.678942,4.0,17.0,1,8,7,-1,2,2196,0.537341,0.599727,True
8,SHEPPARD WEST,0.735376,1584,12838.0,8.104798,5.0,18.0,1,9,9,0,1,2154,0.699164,0.773909,True
9,DAVISVILLE,0.674429,1477,10583.0,7.165200,5.0,17.0,1,11,10,-1,1,2190,0.639269,0.708687,True


In [355]:
con.sql("""
    SELECT * FROM station_data_final LIMIT 5
""").df()

,station,delay_rate,delays,total_delay,avg_delay,p50,p95,route,adjusted_rank,raw_rank,delta,route_1,scheduled_trips,ci_low,ci_high,worse_than_baseline
0,VAUGHAN METROPOLITAN CENTRE,1.407614,3032,14140.0,4.663588,3.0,9.0,1,1,1,0,1,2154,1.357939,1.458229,True
1,FINCH,1.314748,2924,15404.0,5.268126,4.0,11.0,1,2,2,0,1,2224,1.267086,1.362860,True
2,KENNEDY,1.276182,2754,15916.0,5.779230,4.0,14.0,2,3,3,0,2,2158,1.229831,1.322984,True
3,KIPLING,1.154524,2488,14643.0,5.885450,4.0,13.0,2,4,4,0,2,2155,1.108585,1.200000,True
4,EGLINTON,1.107512,2462,16672.0,6.771730,5.0,15.0,1,5,5,0,1,2223,1.062978,1.149798,True


In [356]:
con.sql("""
    CREATE OR REPLACE TABLE rank_comparison AS
        SELECT station, raw_rank AS rank_value, 'raw' AS rank_type FROM station_data_final 
        UNION ALL
        SELECT station, adjusted_rank AS rank_value, 'adjusted' AS rank_type FROM station_data_final
""")

con.sql("""
    SELECT * FROM rank_comparison
""").df()

con.sql("""
    COPY rank_comparison TO 'output/rank_comparison.csv' (HEADER, DELIMITER ',')
""")
